# Kannolo Dense Index Benchmark (Euclidean)

## Import Required Libraries

In [ ]:
from kannolo import DenseFlatIndex, DensePlainHNSW
import numpy as np
import time
from pathlib import Path

## Load and Prepare Dense Dataset

In [ ]:
# Load dense dataset (adjust path as needed)
data_path = ""
data = np.load(data_path)
print(f"Dataset shape: {data.shape}")
print(f"Data type: {data.dtype}")

# Flatten for indexing
dim = data.shape[1]
data_flat = data.flatten().astype(np.float32)
print(f"Flattened data size: {len(data_flat)}")
n_data = len(data_flat) // dim
print(f"Number of data points: {n_data}, Dimension: {dim}")

print("")

# Load dense queries
queries_path = ""
queries = np.load(queries_path)
print(f"Queries shape: {queries.shape}")
print(f"Queries data type: {queries.dtype}")
queries = queries.flatten().astype(np.float32)
print(f"Flattened queries size: {len(queries)}")
n_queries = len(queries) // dim
print(f"Number of queries: {n_queries}, Dimension: {dim}")

## Create Flat Index and Compute Ground Truth

In [ ]:
# Create flat index from array, metric="euclidean"
flat_index = DenseFlatIndex.build_from_array(data_flat, dim, metric="euclidean")
print("Flat index built")

# Compute ground truth (k=10)
k = 10
results = flat_index.search(queries, k)
groundtruth = results[1] # Take IDs from results
print(f"Ground truth computed for {n_queries} queries, k={k}")

## Build HNSW Index

In [ ]:
# Build HNSW index with n. neighbors m=32 and construction precision ef_construction=200
start = time.time()
hnsw = DensePlainHNSW.build_from_array(data_flat, dim, metric="euclidean", m=32, ef_construction=200)
end = time.time()
print(f"HNSW index built in {end - start} seconds")

## Search with Multiple Parameter Combinations

In [ ]:
# Parameter configurations to test
configs = [
    {"ef_search": 20, "early_exit_threshold": None},
    {"ef_search": 100, "early_exit_threshold": None},
    {"ef_search": 15, "early_exit_threshold": 0.1}
]

results = []

for config in configs:
    ef_search = config["ef_search"]
    early_exit = config["early_exit_threshold"]
    
    # Search
    start = time.time()
    distances, results_ids = hnsw.search(queries, k, ef_search=ef_search, early_exit_threshold=early_exit)
    elapsed = time.time() - start
    
    # Compute accuracy
    matches = 0
    for i in range(n_queries):
        gt_set = set(groundtruth[i*k:(i+1)*k])  # Ground truth for this query
        result_set = set(results_ids[i*k:(i+1)*k])  # Result for this query
        matches += len(gt_set & result_set)
    accuracy = (matches / (n_queries * k)) * 100
    avg_time = (elapsed / n_queries) * 10**6  # us per query
    
    results.append({
        "ef_search": ef_search,
        "early_exit_threshold": early_exit,
        "avg_time_ms": avg_time,
        "accuracy": accuracy
    })
    
    print(f"ef_search={ef_search}, early_exit={early_exit}: "
          f"avg_time={avg_time:.3f}us, accuracy={accuracy:.1f}%")

print("\n" + "="*60)
for r in results:
    print(f"Config: ef_search={r['ef_search']}, early_exit={r['early_exit_threshold']}")
    print(f"  Avg Query Time: {r['avg_time_ms']:.3f} us")
    print(f"  Accuracy: {r['accuracy']:.1f}%")
    print()